# 02 - Geracao de variacoes filtradas dos dados interim

Este notebook cria subdatasets em `data/interim` a partir dos Parquets de `items-notas-fiscais`. As variacoes sao gravadas antes do preprocessamento, para que o notebook `03_interim_data_exploration.ipynb` possa ser executado novamente apontando para cada base filtrada via `DATASET_NAME`.

As saidas contem apenas linhas de item, identificadas pela presenca da descricao do produto/servico.

In [14]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_ROOT = PROJECT_ROOT / "data" / "interim"
SOURCE_DATASET_NAME = "items-notas-fiscais"
SOURCE_ITEMS_DIR = INTERIM_ROOT / SOURCE_DATASET_NAME

NCM_OUTPUT_DATASET_NAME = "items-notas-fiscais-ncm-prefixos"
EMITENTES_OUTPUT_DATASET_NAME = "items-notas-fiscais-emitentes"

ALLOW_OVERWRITE = False

SOURCE_ITEMS_DIR

WindowsPath('c:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais')

## Configuracao dos filtros

### NCMs
NCMs sao tratados como prefixos; por exemplo, `"22"` mantem todos os NCMs iniciados por `22`, enquanto `"2203"` restringe mais.

In [15]:
NCM_PREFIXES = [

    # Carnes
    "0201",  # Carne bovina fresca/refrigerada
    "0203",  # Carne suína
    "0207",  # Carne de aves

    # Peixes
    "0302",  # Peixes frescos/refrigerados
    "0303",  # Peixes congelados
    "0304",  # Filés de peixes
    "0305",  # Peixes secos/salgados/defumados

    # Leite, laticínios e ovos
    "0401",  # Leite e creme de leite
    "0402",  # Leite concentrado/em pó
    "0403",  # Iogurte e leite fermentado
    "0405",  # Manteiga
    "0406",  # Queijos e requeijão
    "0407",  # Ovos
    "0409",  # Mel

    # Hortaliças e legumes
    "0701",  # Batatas
    "0703",  # Cebola, alho etc.
    "0704",
    "0705",
    "0706",
    "0707",
    "0708",
    "0709",
    "0710",  # Legumes congelados
    "0713",  # Feijão e outras leguminosas
    "0714",  # Mandioca, batata-doce etc.

    # Frutas
    "0801",
    "0802",
    "0803",
    "0804",
    "0805",
    "0806",
    "0807",
    "0808",
    "0809",
    "0810",
    "0811",  # Frutas congeladas
    "0813",  # Frutas secas

    # Café, chá e temperos
    "0901",  # Café
    "0902",  # Chá
    "0904",
    "0905",
    "0906",
    "0907",
    "0908",
    "0909",
    "0910",  # Especiarias

    # Cereais
    "1004",  # Aveia
    "1005",  # Milho
    "1006",  # Arroz

    # Farinhas, flocos e derivados
    "1101",
    "1102",
    "1103",
    "1104",
    "1105",
    "1106",

    # Óleos e gorduras
    "1507",  # Óleo de soja
    "1509",  # Azeite de oliva
    "1512",  # Óleo de girassol etc.
    "1517",  # Margarina

    # Embutidos e conservas de carnes/peixes
    "1601",  # Linguiças, salsichas etc.
    "1602",  # Carnes preparadas
    "1604",  # Peixes preparados/conservados

    # Açúcar e doces
    "1701",  # Açúcar
    "1704",  # Balas e confeitos

    # Chocolate e cacau
    "1805",  # Cacau em pó
    "1806",  # Chocolates

    # Massas, cereais, pães e biscoitos
    "1901",
    "1902",  # Massas
    "1904",  # Cereais matinais
    "1905",  # Pães, biscoitos, bolachas etc.

    # Conservas, frutas e vegetais preparados
    "2001",
    "2002",
    "2003",
    "2004",
    "2005",
    "2007",  # Geleias e doces
    "2008",  # Frutas preparadas
    "2009",  # Sucos

    # Preparações alimentícias
    "2101",  # Café solúvel etc.
    "2103",  # Molhos e temperos
    "2104",  # Sopas e caldos
    "2105",  # Sorvetes
    "2106",  # Outras preparações alimentícias

    # Bebidas
    "2201",  # Água
    "2202",  # Refrigerantes e bebidas não alcoólicas
    "2203",  # Cerveja
    "2209",  # Vinagre

    # Sal
    "2501",

    # Higiene pessoal
    "3304",  # Produtos para pele
    "3305",  # Shampoo e produtos para cabelo
    "3306",  # Creme dental e higiene bucal
    "3307",  # Desodorantes e produtos de higiene

    # Sabões e limpeza
    "3401",  # Sabonetes e sabões
    "3402",  # Detergentes e produtos de limpeza
    "3405",  # Produtos de limpeza/polimento

    # Inseticidas e desinfetantes
    "3808",

    # Utensílios domésticos de plástico
    "3924",

    # Papel higiênico, guardanapos, papel-toalha etc.
    "4818",

    # Alimentos para animais
    "2309",

    # Fósforos
    "3605",

    # Pilhas
    "8506",

    # Lâmpadas
    "8539",

    # Escovas, vassouras e similares
    "9603",

    # Fraldas, absorventes etc.
    "9619",
]

### Emitentes

Filtro de Emitentes permite filtrar estabelecimentos especificos para a análise

In [16]:
EMITENTES = [
    # "RAZAO SOCIAL DO EMITENTE",
]

In [17]:
def find_col(columns, *patterns):
    """Return the first column whose uppercase name contains all informed patterns."""
    normalized = {col: str(col).upper() for col in columns}
    for col, upper in normalized.items():
        if all(pattern.upper() in upper for pattern in patterns):
            return col
    raise KeyError(f"Nenhuma coluna encontrada para os padroes: {patterns}")


def normalize_text(value):
    """Normalize text for stable equality comparisons."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_ncm(series):
    """Normalize NCM values to digit-only strings without the float suffix."""
    ncm = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    ncm = ncm.str.replace(r"\D", "", regex=True)
    ncm = ncm.mask(ncm.eq(""), pd.NA)
    return ncm.str.zfill(8)


def normalize_ncm_prefixes(prefixes):
    """Normalize configured NCM prefixes to digit-only strings."""
    normalized = []
    for prefix in prefixes:
        text = re.sub(r"\D", "", str(prefix).strip())
        if text:
            normalized.append(text)
    return sorted(set(normalized))


def ensure_can_write(output_dir):
    """Raise when output Parquets already exist and overwrite is disabled."""
    existing = sorted(output_dir.glob("*.parquet")) if output_dir.exists() else []
    if existing and not ALLOW_OVERWRITE:
        files = ", ".join(path.name for path in existing[:5])
        raise FileExistsError(
            f"A saida {output_dir} ja contem Parquets ({files}). "
            "Use outro nome de dataset ou remova os arquivos manualmente se quiser regenerar."
        )


def ensure_unique_csv_can_write(dataset_name):
    """Raise when the unique products CSV already exists and overwrite is disabled."""
    output_path = INTERIM_ROOT / "items-unicos" / dataset_name / "produtos-unicos.csv"
    if output_path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(
            f"O CSV de produtos unicos ja existe: {output_path}. "
            "Use outro nome de dataset ou remova o arquivo manualmente se quiser regenerar."
        )


def write_unique_products(filtered_frames, dataset_name, desc_col):
    """Write the unique product descriptions CSV for a filtered dataset."""
    unique_dir = INTERIM_ROOT / "items-unicos" / dataset_name
    output_path = unique_dir / "produtos-unicos.csv"
    descriptions = pd.concat(
        [frame[desc_col].astype("string").str.strip() for frame in filtered_frames],
        ignore_index=True,
    )
    unique_products = (
        descriptions.dropna()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .sort_values(kind="mergesort")
        .rename("nome_produto")
        .to_frame()
    )

    unique_dir.mkdir(parents=True, exist_ok=True)
    unique_products.to_csv(output_path, index=False)
    return output_path, len(unique_products)


def generate_filtered_dataset(dataset_name, filter_label, filter_func):
    """Generate one filtered interim dataset, preserving monthly Parquet file names."""
    output_dir = INTERIM_ROOT / dataset_name
    ensure_can_write(output_dir)
    ensure_unique_csv_can_write(dataset_name)

    output_dir.mkdir(parents=True, exist_ok=True)
    summaries = []
    filtered_frames = []

    for parquet_file in parquet_paths:
        frame = pd.read_parquet(parquet_file)
        item_frame = frame[frame[desc_col].notna()].copy()
        filtered = item_frame[filter_func(item_frame)].copy()
        filtered.to_parquet(output_dir / parquet_file.name, index=False)
        filtered_frames.append(filtered)
        summaries.append(
            {
                "dataset": dataset_name,
                "filtro": filter_label,
                "arquivo": parquet_file.name,
                "linhas_origem": len(frame),
                "itens_origem": len(item_frame),
                "itens_filtrados": len(filtered),
            }
        )

    unique_path, unique_count = write_unique_products(filtered_frames, dataset_name, desc_col)
    summary = pd.DataFrame(summaries)
    summary["produtos_unicos_csv"] = str(unique_path)
    summary["produtos_unicos"] = unique_count
    return summary

## Inspecao da base de origem

Use estas tabelas para escolher prefixos de NCM e nomes de emitentes antes de preencher a configuracao.

In [18]:
parquet_paths = sorted(SOURCE_ITEMS_DIR.glob("*.parquet"))
if not parquet_paths:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {SOURCE_ITEMS_DIR}")

sample = pd.read_parquet(parquet_paths[0])
desc_col = find_col(sample.columns, "PRODUTO/SERV")
ncm_col = find_col([col for col in sample.columns if "TIPO" not in str(col).upper()], "NCM/SH")
emitente_col = find_col(sample.columns, "SOCIAL", "EMITENTE")

pd.Series(
    {
        "arquivos_origem": len(parquet_paths),
        "descricao": desc_col,
        "ncm": ncm_col,
        "emitente": emitente_col,
    },
    name="configuracao_detectada",
)

arquivos_origem                               4
descricao          DESCRIÇÃO DO PRODUTO/SERVIÇO
ncm                               CÓDIGO NCM/SH
emitente                  RAZÃO SOCIAL EMITENTE
Name: configuracao_detectada, dtype: object

In [19]:
preview_frames = []
for parquet_file in parquet_paths:
    frame = pd.read_parquet(parquet_file, columns=[desc_col, ncm_col, emitente_col])
    frame = frame[frame[desc_col].notna()].copy()
    frame["ncm_str"] = normalize_ncm(frame[ncm_col])
    frame["emitente_normalizado"] = frame[emitente_col].map(normalize_text)
    preview_frames.append(frame)

preview = pd.concat(preview_frames, ignore_index=True)
preview.shape

(1765015, 5)

### Distribuição dos NCMs

In [20]:
ncm_preview = (
    preview.groupby("ncm_str", dropna=False)
    .agg(
        itens=(desc_col, "size"),
        descricoes_unicas=(desc_col, "nunique"),
    )
    .reset_index()
    .sort_values("itens", ascending=False)
)
ncm_preview.head(30)

,ncm_str,itens,descricoes_unicas
3241,49019900,158207,27285
6268,90189099,95769,15731
6278,90211020,53663,15657
6285,90213190,23818,3929
2571,38221990,21662,8813
6240,90183929,20036,5141
413,07099990,18867,1276
3256,49119900,16952,478
6043,87089990,16117,9017
5020,84439933,15032,2278


### Distribuição dos Emitentes

In [21]:
emitentes_preview = (
    preview.groupby([emitente_col, "emitente_normalizado"], dropna=False)
    .agg(
        itens=(desc_col, "size"),
        descricoes_unicas=(desc_col, "nunique"),
        ncms_unicos=("ncm_str", "nunique"),
    )
    .reset_index()
    .sort_values("itens", ascending=False)
)
emitentes_preview.head(50)

,RAZÃO SOCIAL EMITENTE,emitente_normalizado,itens,descricoes_unicas,ncms_unicos
15349,EMPRES BRASILEIRA DE CORREIOS E TELEGRAFOS,EMPRES BRASILEIRA DE CORREIOS E TELEGRAFOS,77760,1860,2
7396,BRS SUPRIMENTOS CORPORATIVOS S/A,BRS SUPRIMENTOS CORPORATIVOS S/A,77530,2720,436
15394,ENDOLOG LOGISTICA E ARMAZENS LTDA,ENDOLOG LOGISTICA E ARMAZENS LTDA,51289,2648,40
5920,AUTOPEL AUTOMACAO COMERCIAL E INFORMATICA LTDA.,AUTOPEL AUTOMACAO COMERCIAL E INFORMATICA LTDA.,31347,1224,234
42474,TRAUMINAS DIST MAT CIRURG HOSP LTDA,TRAUMINAS DIST MAT CIRURG HOSP LTDA,23947,4258,9
6765,BIO 2 IMP. E COM DE MAT. MED. HOSP. LTDA,BIO 2 IMP. E COM DE MAT. MED. HOSP. LTDA,17063,7836,18
8333,CASA DA MOEDA DO BRASIL,CASA DA MOEDA DO BRASIL,16091,41,13
14135,EDITORA FTD S.A.,EDITORA FTD S.A.,12837,387,1
40235,SIMPRESS COMERCIO LOCACAO E SERVICOS LTDA,SIMPRESS COMERCIO LOCACAO E SERVICOS LTDA,12174,352,44
14161,EDITORA MODERNA LTDA.,EDITORA MODERNA LTDA.,11697,422,1


## Validacao dos filtros configurados

In [22]:
normalized_ncm_prefixes = normalize_ncm_prefixes(NCM_PREFIXES)
normalized_emitentes = []
for value in EMITENTES:
    normalized = normalize_text(value)
    if not pd.isna(normalized) and normalized:
        normalized_emitentes.append(normalized)
normalized_emitentes = sorted(set(normalized_emitentes))

configured_filters = pd.DataFrame(
    {
        "filtro": ["NCM_PREFIXES", "EMITENTES"],
        "valores_normalizados": [normalized_ncm_prefixes, normalized_emitentes],
        "quantidade": [len(normalized_ncm_prefixes), len(normalized_emitentes)],
    }
)
configured_filters

,filtro,valores_normalizados,quantidade
0,NCM_PREFIXES,"[0201, 0203, 0207, 0302, 0303, 0304, 0305, 0401, 0402, 0403, 0405, 0406, 0407, 0409, 0701, 0703, 0704, 0705, 0706, 0...",104
1,EMITENTES,[],0


In [23]:
if normalized_ncm_prefixes:
    mask_ncm_preview = preview["ncm_str"].fillna("").apply(lambda value: any(value.startswith(prefix) for prefix in normalized_ncm_prefixes))
    display(pd.DataFrame({"filtro": ["NCM"], "itens_previstos": [int(mask_ncm_preview.sum())]}))
else:
    print("NCM_PREFIXES esta vazio; a base por NCM nao sera gerada.")

if normalized_emitentes:
    mask_emitentes_preview = preview["emitente_normalizado"].isin(normalized_emitentes)
    display(pd.DataFrame({"filtro": ["emitentes"], "itens_previstos": [int(mask_emitentes_preview.sum())]}))
else:
    print("EMITENTES esta vazio; a base por emitentes nao sera gerada.")

,filtro,itens_previstos
0,NCM,483001


EMITENTES esta vazio; a base por emitentes nao sera gerada.


## Geracao das bases filtradas

As celulas abaixo gravam os Parquets mensais e os CSVs de produtos unicos. Se os arquivos de saida ja existirem, a execucao falha por padrao.

In [26]:
summaries = []

if normalized_ncm_prefixes:
    def ncm_filter(frame):
        ncm = normalize_ncm(frame[ncm_col]).fillna("")
        return ncm.apply(lambda value: any(value.startswith(prefix) for prefix in normalized_ncm_prefixes))

    summaries.append(generate_filtered_dataset(NCM_OUTPUT_DATASET_NAME, "ncm_prefixos", ncm_filter))

if normalized_emitentes:
    def emitente_filter(frame):
        emitentes = frame[emitente_col].map(normalize_text)
        return emitentes.isin(normalized_emitentes)

    summaries.append(generate_filtered_dataset(EMITENTES_OUTPUT_DATASET_NAME, "emitentes", emitente_filter))

if not summaries:
    raise ValueError("Preencha NCM_PREFIXES ou EMITENTES antes de gerar bases filtradas.")

generation_summary = pd.concat(summaries, ignore_index=True)
generation_summary

,dataset,filtro,arquivo,linhas_origem,itens_origem,itens_filtrados,produtos_unicos_csv,produtos_unicos
0,items-notas-fiscais-ncm-prefixos,ncm_prefixos,2025-01.parquet,478562,369683,91543,c:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-unicos\items-notas-fiscais-ncm-prefixos\produtos-unicos.csv,73786
1,items-notas-fiscais-ncm-prefixos,ncm_prefixos,2025-02.parquet,608847,490657,112912,c:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-unicos\items-notas-fiscais-ncm-prefixos\produtos-unicos.csv,73786
2,items-notas-fiscais-ncm-prefixos,ncm_prefixos,2025-03.parquet,551362,434549,134515,c:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-unicos\items-notas-fiscais-ncm-prefixos\produtos-unicos.csv,73786
3,items-notas-fiscais-ncm-prefixos,ncm_prefixos,2025-04.parquet,598767,470126,144031,c:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-unicos\items-notas-fiscais-ncm-prefixos\produtos-unicos.csv,73786


## Como analisar as variacoes

Depois de gerar as bases, execute o notebook `03_interim_data_exploration.ipynb` apontando para a variacao desejada:

```powershell
$env:DATASET_NAME = "items-notas-fiscais-ncm-prefixos"
jupyter notebook notebooks/03_interim_data_exploration.ipynb
```

ou:

```powershell
$env:DATASET_NAME = "items-notas-fiscais-emitentes"
jupyter notebook notebooks/03_interim_data_exploration.ipynb
```